Awez Shaik

## 1. Check CUDA availability

In [ ]:
import torch

# Use the GPU if CUDA is available, otherwise fall back to the CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

## 2. Load FashionMNIST and split into train / validation

In [ ]:
import torch
from torch.utils.data import random_split
from torchvision import datasets
from torchvision.transforms import v2

# Convert images to float32 tensors scaled to [0, 1], then normalize with FashionMNIST mean/std
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.2860,), (0.3530,)),
])

# Download FashionMNIST (60,000 training images, 10,000 test images)
full_train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

# Split the 60,000 training images into 55,000 train / 5,000 validation
train_dataset, val_dataset = random_split(
    full_train_dataset, [55000, 5000],
    generator=torch.Generator().manual_seed(42),
)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

## 3. DataLoaders (batch size 32) and first training image

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

# Shuffle only the training data so each epoch sees batches in a new order
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Inspect the first image in the training set
first_image, first_label = train_dataset[0]

print(f"First image shape: {first_image.shape}")  # expected [1, 28, 28]
print(f"First image dtype: {first_image.dtype}")  # expected torch.float32
print(f"First image label: {first_label}")

## 4. Model: `ImageClassifier`

In [ ]:
from torch import nn

class ImageClassifier(nn.Module):
    def __init__(self, n_features, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features, n_hidden1),  # input -> hidden layer 1
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),  # hidden layer 1 -> hidden layer 2
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),  # hidden layer 2 -> output, no activation (raw logits)
        )

    def forward(self, x):
        return self.layers(x)

# FashionMNIST: 28x28 grayscale images, 10 clothing classes
model = ImageClassifier(n_features=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10).to(device)
print(model)

## 5. Training loop (SGD + torchmetrics accuracy)

In [ ]:
import torch
import torchmetrics
from torch import nn

N_EPOCHS = 20
LEARNING_RATE = 0.01
MOMENTUM = 0.9
N_CLASSES = 10

# CrossEntropyLoss applies softmax internally, so the model outputs raw logits
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

# Separate metric objects so training and validation accumulate independently
train_metric = torchmetrics.Accuracy(task="multiclass", num_classes=N_CLASSES).to(device)
val_metric = torchmetrics.Accuracy(task="multiclass", num_classes=N_CLASSES).to(device)

def evaluate(model, data_loader, metric):
    """Return the metric computed over the whole data loader."""
    model.eval()
    metric.reset()
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            metric.update(model(images), labels)
    return metric.compute().item()

# Per-epoch history, used later to plot accuracy
history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(N_EPOCHS):
    model.train()
    train_metric.reset()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_metric.update(logits, labels)

    train_loss = running_loss / len(train_loader)
    train_acc = train_metric.compute().item()
    val_acc = evaluate(model, val_loader, val_metric)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch + 1}/{N_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"train accuracy: {train_acc:.4f} | "
        f"valid accuracy: {val_acc:.4f}"
    )

## 6. Plot accuracy

In [ ]:
import matplotlib.pyplot as plt

TRAIN_COLOR = "#2a78d6"  # blue
VAL_COLOR = "#eb6834"  # orange

epochs = range(1, len(history["train_acc"]) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, history["train_acc"], color=TRAIN_COLOR, linewidth=2,
        marker="o", markersize=5, label="Training")
ax.plot(epochs, history["val_acc"], color=VAL_COLOR, linewidth=2,
        marker="s", markersize=5, label="Validation")

# Label the final value of each line directly
ax.annotate(f"{history['train_acc'][-1]:.3f}", (epochs[-1], history["train_acc"][-1]),
            xytext=(6, 0), textcoords="offset points", va="center")
ax.annotate(f"{history['val_acc'][-1]:.3f}", (epochs[-1], history["val_acc"][-1]),
            xytext=(6, 0), textcoords="offset points", va="center")

ax.set_title("FashionMNIST accuracy per epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_xticks(list(epochs))
ax.grid(True, color="#e0e0e0", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig("accuracy.png", dpi=150)
plt.show()